In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import contextily as cx
from adjustText import adjust_text
import itertools


from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.nh_run import continue_run
from neuralhydrology.utils.config import Config
from neuralhydrology.evaluation import get_tester, metrics

In [2]:
# -------- Paths ---------
CONFIG_PATH = Path("./basins_excluded.yml")
RUNS_DIR = Path("runs")
PLOTS_DIR = Path("./evaluation_plots")
RUNS_DIR = Path("runs")
PLOTS_DIR = Path("evaluation_plots")
ATTRIBUTES_FILE= Path('./data/attributes/attributes_other.csv')

In [3]:
precip_products = [
    "era5land_total_precipitation",
    "chirps_precipitation",
    "mswep_precipitation"
]

In [5]:
base_config_path = Path("basins_excluded.yml")

with open(base_config_path, "r") as f:
    base_config = yaml.safe_load(f)

In [7]:
# for r in range(1, len(precip_products) + 1):
#     for precip_combo in itertools.combinations(precip_products, r):
#         config = base_config.copy()

#         # Update dynamic inputs:
#         # Keep temperature and radiation, replace precip
#         config["dynamic_inputs"] = [
#             "surface_net_solar_radiation_mean",
#             "temperature_2m_max",
#             "temperature_2m_min",
#             *precip_combo
#         ]

#         # Create unique experiment name
#         precip_name = "_".join(precip_combo)
#         config["experiment_name"] = (
#             f"precip_{precip_name}"
#         )

#         # Save temporary config file
#         temp_config_path = Path(f"temp_{precip_name}")
#         with open(temp_config_path, "w") as f:
#             yaml.dump(config, f)

#         print(f"Running: {config['experiment_name']}")

#         # Run training
#         if torch.cuda.is_available() or torch.backends.mps.is_available():
#             start_run(config_file=temp_config_path)
#         else:
#             start_run(config_file=temp_config_path, gpu=-1)

In [28]:
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
if torch.cuda.is_available() or torch.backends.mps.is_available():
    start_run(config_file=Path("basins_excluded.yml"))

# fall back to CPU-only mode
else:
    start_run(config_file=Path("basins_excluded.yml"), gpu=-1)

2026-03-03 16:24:42,916: Logging to /home/azureuser/sky_workdir/extending_caravan/runs/caravan_chirps_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0303_162442/output.log initialized.
2026-03-03 16:24:42,917: ### Folder structure created at /home/azureuser/sky_workdir/extending_caravan/runs/caravan_chirps_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0303_162442
2026-03-03 16:24:42,917: ### Run configurations for caravan_chirps_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111
2026-03-03 16:24:42,918: experiment_name: caravan_chirps_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111
2026-03-03 16:24:42,918: run_dir: /home/azureuser/sky_workdir/extending_caravan/runs/caravan_chirps_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0303_162442
2026-03-03 16:24:42,918: train_basin_file: basins_without_snow.txt
2026-03-03 16:24:42,919: validation_basin_file: basins_without_snow.txt
2026-03-03 16:24:42,920: te

In [4]:
# finetune

In [3]:
finetune(Path("finetune.yml"))

2026-03-03 19:31:53,946: Logging to /home/azureuser/sky_workdir/extending_caravan/runs/finetune_UY_basins_seq_180_0303_193153/output.log initialized.
2026-03-03 19:31:53,946: ### Folder structure created at /home/azureuser/sky_workdir/extending_caravan/runs/finetune_UY_basins_seq_180_0303_193153
2026-03-03 19:31:53,947: ### Start finetuning with pretrained model stored in runs/caravan_and_uy_chirps_and_mswep_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0203_030221
2026-03-03 19:31:53,947: ### Run configurations for finetune_UY_basins_seq_180
2026-03-03 19:31:53,948: batch_size: 256
2026-03-03 19:31:53,948: clip_gradient_norm: 1
2026-03-03 19:31:53,949: commit_hash: None
2026-03-03 19:31:53,949: data_dir: data
2026-03-03 19:31:53,950: dataset: generic
2026-03-03 19:31:53,950: device: cuda:0
2026-03-03 19:31:53,950: dynamic_inputs: ['temperature_2m_max', 'temperature_2m_min', 'surface_net_solar_radiation_mean', 'chirps_precipitation', 'mswep_precipitation']
2026-03-03

/home/azureuser/miniconda3/envs/neuralhydrology/lib/python3.10/site-packages/neuralhydrology/training/basetrainer.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.

2026-03-03 19:31:57,589: Validation set to validate every 1 epoch(s), but 'validate_n_random_basins' not set or set to zero. Will validate on the entire validation set.
# Epoch 1: 100%|██████████| 132/132 [00:05<00:00, 25.57it/s, Loss: 0.0159]
2026-03-03 19:32:02,759: Epoch 1 average loss: avg_loss: 0.02788, avg_total_loss: 0.02788
# Validation: 100%|██████████| 11/11 [00:09<00:00,  1.17it/s]
2026-03-03 19:32:12,212: Stored metrics at /home/azureuser/sky_workdir/extending_caravan/runs/finetune_UY_basins_seq_180_0303_193153/validation/model_epoch001/validation_metrics.csv
2026-03-03 19:32:12,216: Stored results at /home/azureuser/sky_workdir/extending_caravan/runs/finetune_UY_basins_seq_180_0303_193153/validation/model_epoch001/validation_results.p
2026-03-03 19:32:12,218: Epoch 1 average validation loss: 0.06387 -- Median validation metrics: avg_loss: 0.06387, NSE: 0.48617, KGE: 0.63546, Alpha-NSE: 1.15028, Beta-NSE: 0.11488, Pearson-r: 0.82406, RMSE: 1.81275, MSE: 3.28607, Beta-KGE: 1